In [1]:
import os
import shutil
import subprocess
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
    reset_activity_context,
    visualize_conversation,
)

from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient

# 02 - 관측 가능성(Observability) 에이전트

앞선 노트북들에서 기본 리서치 에이전트와 Chief of Staff 멀티에이전트 프레임워크를 만들어 봤습니다. 이미 강력한 에이전트들이지만 할 수 있는 일에는 여전히 한계가 있었습니다. 웹 검색 에이전트는 인터넷 검색에 묶여 있었고, Chief of Staff 에이전트는 자기 파일 시스템과 상호작용하는 데 그쳤습니다.

이는 심각한 제약입니다. 실제 에이전트는 데이터베이스, API, 파일 시스템, 그 밖의 전문 서비스 같은 다른 시스템과 상호작용해야 하는 경우가 많습니다. [MCP(Model Context Protocol)](https://modelcontextprotocol.io/docs/getting-started/intro)는 AI와 도구를 연결하기 위한 오픈소스 표준으로, 에이전트와 이런 외부 시스템을 손쉽게 이어 줍니다. 이 노트북에서는 MCP 서버를 에이전트에 연결하는 방법을 살펴봅니다.

**MCP에 대한 자세한 내용이 필요하신가요?** 전체 설정 방법, 구성 모범 사례, 문제 해결 팁은 [Claude Code MCP 문서](https://docs.claude.com/en/docs/claude-code/mcp)를 참고하세요.

## MCP 서버 소개
### 1. Git MCP 서버

먼저 에이전트에 Git 저장소를 이해하고 다룰 수 있는 능력을 줘 보겠습니다. [Git MCP 서버](https://github.com/modelcontextprotocol/servers/tree/main/src/git)를 에이전트에 추가하면 Git 전용 도구 13개를 사용할 수 있게 되어, 커밋 이력을 살펴보고, 파일 변경을 확인하고, 브랜치를 만들고, 심지어 커밋까지 할 수 있습니다. 수동적인 관찰자였던 에이전트가 개발 워크플로의 능동적인 참여자로 바뀌는 것이죠. 이 예제에서는 Git 도구만으로 저장소의 이력을 탐색하도록 에이전트를 설정합니다. 아주 단순하지만, 이것만 알면 풀 리퀘스트를 자동으로 만들거나, 코드 변천 패턴을 분석하거나, 여러 저장소에 걸친 복잡한 Git 워크플로를 관리하는 에이전트를 상상하기는 어렵지 않습니다.

In [2]:
# Get the git repository root (mcp_server_git requires a valid git repo path)
# os.getcwd() may return a subdirectory, so we find the actual repo root
git_executable = shutil.which("git")
if git_executable is None:
    raise RuntimeError("Git executable not found in PATH")

git_repo_root = subprocess.run(  # noqa: S603
    [git_executable, "rev-parse", "--show-toplevel"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

# Define our git MCP server (installed via uv sync from pyproject.toml)
git_mcp: dict[str, Any] = {
    "git": {
        "command": "uv",
        "args": ["run", "python", "-m", "mcp_server_git", "--repository", git_repo_root],
    }
}

In [3]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model="claude-opus-4-6",
        mcp_servers=git_mcp,
        allowed_tools=["mcp__git"],
        # disallowed_tools ensures the agent ONLY uses MCP tools, not Bash with git commands
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(
        "Explore this repo's git history and provide a brief summary of recent activity."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__git__git_log()
🤖 Using: mcp__git__git_status()
🤖 Using: mcp__git__git_branch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__git__git_log()
🤖 Using: mcp__git__git_status()
🤖 Using: mcp__git__git_branch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


In [4]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
## Git Repository Summary

### Current Branch
You're on the **`upstream-contribution`** branch (up to date with origin), with `main` also available locally.

---

### Recent Commit Activity (Last ~5 Days)

| Date | Author | Summary |
|------|--------|---------|
| **Nov 27, 2025** | costiash | 3 commits enhancing the **Claude Agent SDK** - improved chief of staff agent, notebooks, observability agent, research agent, documentation, and utilities |
| **Nov 26, 2025** | Pedram Navid | Added GitHub issue templates, `/review-issue` command, `/add-registry` slash command, and new cookbook entries |
| **Nov 25, 2025** | Elie Schoppik | Renamed PTC notebook to `programmatic_tool_calling_ptc.ipynb` for clarity |
| **Nov 24, 2025** | henrykeetay | Added **tool search cookbook** |
| **Nov 24, 2025** | Alex Notov | Multiple merges consolidating cookbooks for Opus 4.5, dependency updates |
| **Nov 23, 2025** | Cal Rueb | Simplified crop tool notebook with Claude Agent SDK section |
| **Nov 23, 2025** | Pedram Navid | PR comment fixes and lint cleanup |

---

### Key Themes in Recent Development
1. **Claude Agent SDK enhancements** - Major work on agent implementations (research, chief of staff, observability agents)
2. **New cookbooks** - Tool search, crop tool, programmatic tool calling
3. **CI/CD improvements** - PR review workflows, issue templates, slash commands
4. **Documentation** - Added troubleshooting guides, codebase overviews

---

### Working Directory Status
There are **uncommitted changes** in your working directory:
- **22 modified files** (mostly in `claude_agent_sdk/`)
- **4 deleted files** (documentation files in `docs/`)
- **6 untracked files** (new reports, plans, VS Code config)

These changes appear to be further work on the Claude Agent SDK agents, notebooks, and utilities that haven't been staged or committed yet.

### 2. GitHub MCP 서버

이제 로컬 Git 작업에서 GitHub 플랫폼 전체 연동으로 한 단계 올라가 보겠습니다. [공식 GitHub MCP 서버](https://github.com/github/github-mcp-server/tree/main)로 바꾸면 에이전트가 GitHub 생태계 전반과 상호작용하는 100개가 넘는 도구를 쓸 수 있게 됩니다. 이슈와 풀 리퀘스트 관리부터 CI/CD 워크플로 모니터링, 코드 보안 경고 분석까지 가능합니다. 이 서버는 공개 저장소와 비공개 저장소 모두에서 동작하므로, 보통은 여러 번의 수동 작업이 필요한 복잡한 GitHub 워크플로를 에이전트가 자동화할 수 있습니다.

#### 1단계: GitHub 토큰 준비하기

GitHub 개인 액세스 토큰이 필요합니다. [여기](https://github.com/settings/personal-access-tokens/new)에서 발급받아 .env 파일에 ```GITHUB_TOKEN="<token>"``` 형태로 넣으세요.
> 참고: 토큰을 발급받을 때 기본 옵션(공개 저장소, 계정 권한 없음)으로 "Fine-grained" 토큰을 선택하는 것이 이 데모를 돌리는 가장 쉬운 방법입니다.

또한 이 예제를 실행하려면 머신에서 [Docker](https://www.docker.com/products/docker-desktop/)가 돌고 있어야 합니다. GitHub MCP 서버가 보안과 격리를 위해 컨테이너 환경에서 실행되기 때문입니다.

**Docker 빠른 설정:**
- [docker.com](https://www.docker.com/products/docker-desktop/)에서 Docker Desktop을 설치하세요
- Docker가 실행 중인지 확인하세요(시스템 트레이에 Docker 아이콘이 보입니다)
- 터미널에서 `docker --version`으로 확인하세요
- **문제 해결:** Docker가 시작되지 않으면 BIOS에서 가상화가 켜져 있는지 확인하세요. 자세한 설정 방법은 [Docker 문서](https://docs.docker.com/get-docker/)를 참고하세요

#### 2단계: MCP 서버를 정의하고 에이전트 루프를 시작하세요!

In [5]:
# define our github mcp server
load_dotenv(override=True)
github_mcp: dict[str, Any] = {
    "github": {
        "command": "docker",
        "args": [
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": os.environ.get("GITHUB_TOKEN")},
    }
}

In [6]:
# run our agent
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model="claude-opus-4-6",
        mcp_servers=github_mcp,
        allowed_tools=["mcp__github"],
        # disallowed_tools ensures the agent ONLY uses MCP tools, not Bash with gh CLI
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(
        "Search for the anthropics/claude-agent-sdk-python repository and give me a few key facts about it."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__github__search_repositories()
✓ Tool completed
🤖 Thinking...


In [7]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
Here are the key facts about the **anthropics/claude-agent-sdk-python** repository:

| Fact | Details |
|------|---------|
| **Full Name** | anthropics/claude-agent-sdk-python |
| **URL** | https://github.com/anthropics/claude-agent-sdk-python |
| **Language** | Python |
| **Stars** | ⭐ 3,357 |
| **Forks** | 🍴 435 |
| **Open Issues** | 149 |
| **Created** | June 11, 2025 |
| **Last Updated** | December 4, 2025 |
| **Default Branch** | main |
| **Visibility** | Public |
| **Archived** | No |

This is the official Python SDK for building Claude agents, maintained by Anthropic. It's quite popular with over 3,300 stars and has an active community with 435 forks. The repository is actively maintained (recently updated) and has a notable number of open issues (149), which suggests active development and community engagement.

## 실제 사용 사례: 관측 가능성 에이전트

이 정도의 간단한 설정만으로도 이미 자가 치유 소프트웨어 시스템처럼 동작하는 에이전트를 가질 수 있습니다!

In [8]:
load_dotenv(override=True)

prompt = """Analyze the CI health for facebook/react repository.

Examine the most recent runs of the 'CI' workflow and provide:
1. Current status and what triggered the run (push, PR, schedule, etc.)
2. If failing: identify the specific failing jobs/tests and assess severity
3. If passing: note any concerning patterns (long duration, flaky history)
4. Recommended actions with priority (critical/high/medium/low)

Provide a concise operational summary suitable for an on-call engineer.
Do not create issues or PRs - this is a read-only analysis."""

github_mcp: dict[str, Any] = {
    "github": {
        "command": "docker",
        "args": [
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": os.environ.get("GITHUB_TOKEN")},
    }
}

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model="claude-opus-4-6",
        mcp_servers=github_mcp,
        allowed_tools=["mcp__github"],
        # IMPORTANT: disallowed_tools is required to actually RESTRICT tool usage.
        # Without this, allowed_tools only controls permission prompting, not availability.
        # The agent would still have access to Bash (and could use `gh` CLI instead of MCP).
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(prompt)
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__list_commits()
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__list_pull_requests()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__sea

In [9]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
Based on my comprehensive analysis of the facebook/react repository CI infrastructure, here is the operational summary:

---

# CI Health Analysis: facebook/react

## Executive Summary
**Overall Status: 🟢 HEALTHY**

The React repository's CI appears to be in good health. Recent commits to `main` have been successfully merged, and active PRs show passing CodeSandbox builds.

---

## 1. CI Infrastructure Overview

### Primary Workflows
| Workflow | Trigger | Purpose |
|----------|---------|---------|
| `runtime_build_and_test.yml` | Push to main, PRs | Main CI - builds, tests, Flow checks |
| `shared_lint.yml` | Push to main, PRs | Prettier, ESLint, license checks |
| `compiler_typescript.yml` | PRs touching compiler | Compiler-specific tests |
| `devtools_regression_tests.yml` | PRs | DevTools testing |

### Test Matrix Scale
- **90 test shards** (18 configurations × 5 shards each)
- **50 build jobs** (25 workers × 2 release channels)
- **50 test-build shards** (5 configurations × 10 shards)
- Flow checks across multiple inline configs

---

## 2. Recent Main Branch Status

| Commit | Date | Description | Status |
|--------|------|-------------|--------|
| `bf1afad` | Dec 4, 2025 | [react-dom/server] Fix hanging on Deno | ✅ Merged |
| `0526c79` | Dec 3, 2025 | Update changelog with latest releases | ✅ Merged |
| `7dc903c` | Dec 3, 2025 | Patch FlightReplyServer (security fix) | ✅ Merged |
| `36df5e8` | Dec 2, 2025 | Allow building single release channel | ✅ Merged |

**Last 10 commits:** All successfully merged to main, indicating CI is passing.

---

## 3. Active PR CI Status

| PR | Title | CodeSandbox Status |
|----|-------|-------------------|
| #35267 | Fix spelling (behaviour → behavior) | 🟡 Pending (building) |
| #35238 | DevTools navigating commits hotkey | ✅ Success |
| #35287 | Compiler: Fix variable name issue | ✅ Success |
| #35278 | Add DevTools console suppress option | ✅ Success |
| #35226 | Fizz: Push stalled use() to ownerStack | ✅ Success |

---

## 4. Risk Assessment

### ✅ Positive Indicators
- **Main branch stable**: All recent commits merged successfully
- **No open CI failure issues**: Search returned zero CI-related open bugs
- **Active development**: Security patches and features landing regularly
- **PR builds passing**: Most open PRs show successful builds

### ⚠️ Areas to Monitor
- **Large test matrix**: 190+ parallel jobs mean potential for infrastructure flakiness
- **Playwright-based e2e tests**: Browser-based tests can be flaky (Flight fixtures, DevTools e2e)
- **Cache dependencies**: Multiple cache strategies (v6 keys) - cache misses could slow builds

### 📊 CI Complexity Metrics
- ~37KB workflow file for main CI (`runtime_build_and_test.yml`)
- Heavy parallelization with matrix strategies
- Multiple artifact upload/download operations

---

## 5. Recommended Actions

| Priority | Action | Rationale |
|----------|--------|-----------|
| **LOW** | Monitor PR #35267 | Currently building - verify completion |
| **LOW** | No immediate action required | Main branch healthy, PRs passing |
| **INFO** | Security patch merged Dec 3 | PR #35277 fixed critical security vuln in FlightReplyServer - verify downstream impact |

---

## 6. On-Call Notes

**TL;DR for On-Call Engineer:**
- 🟢 **CI is GREEN** - No action required
- Main branch is healthy with successful merges in last 24h
- All checked PRs showing green/passing status
- No open issues flagged for CI failures or flakiness
- Recent security patch (#35277) was successfully merged - monitor for any regressions

**If issues arise:**
1. Check GitHub Actions tab directly: `https://github.com/facebook/react/actions`
2. Key workflows to monitor: "(Runtime) Build and Test", "(Shared) Lint"
3. Caches use `v6` key prefix - if widespread failures, consider cache invalidation

---

*Analysis performed: December 4, 2025*
*Data sources: GitHub API (commits, PRs, status checks, workflow files)*

In [10]:
reset_activity_context()
visualize_conversation(messages)

In [11]:
reset_activity_context()
display_agent_response(messages)

### 모듈로서의 관측 가능성 에이전트

`observability_agent/agent.py` 모듈은 관측 가능성 패턴을 재사용 가능한 `send_query` 함수로 감쌉니다. 내부적으로 `utils.agent_visualizer`의 공용 시각화 유틸리티를 가져다 씁니다.
- **`reset_activity_context()`**: 각 질의가 시작될 때 자동으로 호출됩니다
- **`print_activity()`**: 실행 중 실시간 피드백을 제공합니다
- **`display_agent_response()`**: 최종 결과를 렌더링합니다(`display_result` 파라미터로 제어)

덕분에 최소한의 코드로 이 모듈을 사용할 수 있습니다:

In [12]:
# Reload the module to pick up any changes (useful during development)
from observability_agent.agent import send_query

# The module handles activity display, context reset, and result visualization internally
result = await send_query(
    "Check the CI status for the last 2 runs in anthropics/claude-agent-sdk-python. Just do 3 tool calls, be efficient."
)

🤖 Using: mcp__github__list_commits()
✓ Tool completed
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
✓ Tool completed
✓ Tool completed
🤖 Thinking...


Commit,Message,Date,Status
2437035,chore: bump bundled CLI version to 2.0.58,"Dec 3, 2025 20:09 UTC",⚠️ No CI status available
9809fb6,chore: release v0.1.11 (#383),"Dec 3, 2025 19:42 UTC",⚠️ No CI status available


멀티턴 대화도 매끄럽게 동작합니다. `continue_conversation=True`만 전달하면 됩니다:

In [13]:
# Example 2: Multi-turn conversation for deeper monitoring
result1 = await send_query("What's the current CI status for facebook/react?")

🤖 Using: mcp__github__list_pull_requests()
🤖 Using: mcp__github__list_commits()
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__get_commit()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


PR,Title,Author,CI Status,Updated
#35287,[compiler] Fix JSX variable name issue,@kostya-gromov,🟢 Success,2h ago
#35285,[compiler][poc] Reuse ValidateExhaustiveDeps,@josephsavona,🔵 Draft,6h ago
#35284,[compiler] Fix hoisted primitives bug,@josephsavona,🟢 Success,7h ago
#35282,[compiler] Add effect deps validator,@jackpope,🟢 Success,15h ago
#35281,Improve legacy context warning,@Harshrj53,🟢 Success,20h ago


In [14]:
# Continue the conversation to dig deeper
result2 = await send_query(
    "Are there any flaky tests in the recent failures? You can only make one tool call.",
    continue_conversation=True,
)

🤖 Using: mcp__github__search_issues()
✓ Tool completed
🤖 Thinking...


Metric,Status
Open flaky test issues,0
Recent CI failures,None detected
Test stability,✅ Stable


## 마무리

Claude Code SDK가 Model Context Protocol(MCP)을 통해 외부 시스템과 매끄럽게 연동되는 모습을 살펴봤습니다. Git MCP 서버로 로컬 Git 작업부터 시작해, GitHub 전용 도구 100개 이상을 쓸 수 있는 GitHub 플랫폼 전체 연동까지 점진적으로 확장했습니다. 그 결과 로컬 어시스턴트였던 에이전트가 워크플로를 모니터링하고, CI/CD 실패를 분석하고, 프로덕션 시스템에 대해 실행 가능한 통찰을 제공하는 강력한 관측 가능성 시스템으로 바뀌었습니다.

MCP 서버를 에이전트에 연결함으로써 GitHub Actions 워크플로를 모니터링하고, 실제 실패와 보안 제한을 구분하고, 테스트 실패를 상세히 분석하는 자율 관측 가능성 시스템을 만들었습니다. 이 시스템은 에이전트가 수동적 모니터링에서 지능적 장애 대응으로 나아가며 DevOps 워크플로에 능동적으로 참여할 수 있음을 보여 줍니다.

이것으로 Claude Code SDK 튜토리얼 시리즈의 여정을 일단 마칩니다. 단순한 리서치 에이전트에서 정교한 멀티에이전트 오케스트레이션으로, 그리고 MCP를 통한 외부 시스템 연동까지 나아갔습니다. 이 패턴들을 합치면 거버넌스, 컴플라이언스, 관측 가능성을 유지하면서도 현실의 복잡성을 감당하는 프로덕션 수준 에이전트 시스템의 토대가 됩니다.

### 모든 노트북에서 배운 것

**노트북 00 (리서치 에이전트)에서**
- `query()`와 `ClaudeSDKClient`로 익히는 SDK 핵심 기초
- WebSearch와 Read를 활용한 기본 도구 사용
- 단순한 에이전트 루프와 대화 관리

**노트북 01 (Chief of Staff)에서**
- 고급 기능: 메모리, 출력 스타일, 계획 모드
- 서브에이전트를 통한 멀티에이전트 조율
- 훅과 커스텀 명령을 통한 거버넌스
- 엔터프라이즈 수준의 에이전트 아키텍처

**노트북 02 (관측 가능성 에이전트)에서**
- MCP 서버를 통한 외부 시스템 연동
- 실시간 모니터링과 장애 대응
- 프로덕션 워크플로 자동화
- 확장 가능한 에이전트 배포 패턴

세 에이전트의 완전한 구현은 각각의 디렉터리(`research_agent/`, `chief_of_staff_agent/`, `observability_agent/`)에 있으며, 여러분의 프로덕션 시스템에 통합할 때 참고할 수 있습니다.